# Advanced Chemical Reactor Engineering – Group: Stirred not Shaken
### SDR reactor simulation: glucose → fructose → HMF → FDCA

This notebook completes the **novel reactor** model using a **spinning disc reactor (SDR)** basis.

## Modelling choice used here
To stay consistent with the report text, the SDR is represented as **N ideal CSTRs in series**:
- this gives a **narrower RTD / plug-flow-like behaviour** than a single CSTR,
- while keeping the **same total residence time and chemistry** as the CSTR notebook for a fair comparison,
- and it fits the rotor–stator SDR description from the slides: **multistage, plug flow, high mass transfer**.

The reaction network is kept the same as in the CSTR notebook:
1. Glucose ⇌ Fructose (aqueous)
2. Fructose → HMF (aqueous)
3. HMF → humins / side products (aqueous)
4. HMF + O₂ → FDCA (catalyst / organic phase)

The main reactor-specific changes are therefore:
- **multistage hydrodynamics**,
- **higher SDR mass transfer coefficients**,
- **stagewise solution and profile plots**,
- **rate-determining-step analysis for the SDR base case**.


### 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 6)


### 1. Physical properties

The same fluid-property basis is kept as in the CSTR notebook so the reactor comparison stays consistent.


In [ ]:
T = 160 + 273.15   # K
g = 9.81             # m/s^2
R = 8.3145           # J/mol/K

rho = 780.0          # kg/m^3   density MIBK
mu = 0.35e-3         # Pa.s     dynamic viscosity MIBK
nu = mu / rho        # m^2/s    kinematic viscosity
sigma = 12e-3        # N/m      interfacial tension MIBK/water

# Diffusivities corrected to operating T and viscosity
mu_ref = 0.585e-3
D_HMF  = 6.0e-10 * (T / 298.0) * (mu_ref / mu)   # m^2/s
D_O2   = 3.5e-9  * (T / 298.0) * (mu_ref / mu)   # m^2/s

print(f"T = {T:.2f} K")
print(f"D_HMF = {D_HMF:.3e} m^2/s")
print(f"D_O2  = {D_O2:.3e} m^2/s")


### 2. SDR design basis

For a fair comparison with the CSTR notebook, the **same total reactive volume** and **same total residence time** are used.

The SDR is then approximated as **N stages in series**:
- total reactor volume = 100 m³,
- total residence time = 1 h,
- each stage has volume = total volume / N.

This keeps the chemistry basis unchanged while introducing the narrower RTD expected for the SDR.


In [ ]:
# Overall reactor basis (kept equal to CSTR notebook)
V_total = 100.0                    # m^3 total reactive volume
tau_total = 3600.0                 # s   total mean residence time

# Phase holdups (same as CSTR basis)
eps_g = 0.20
eps_s = 0.10
eps_l = 1.0 - eps_g - eps_s

eps_aq = 1/3
eps_org = 1 - eps_aq

V_g   = eps_g * V_total
V_p   = eps_s * V_total
V_liq = eps_l * V_total
V_aq  = eps_aq * V_liq
V_org = eps_org * V_liq

# Throughputs corresponding to tau_total
F_aq  = V_aq / tau_total          # m^3/s
F_org = V_org / tau_total         # m^3/s

# SDR hydrodynamic representation
N_stages = 20                      # plug-flow approximation
tau_stage = tau_total / N_stages

V_aq_stage  = V_aq  / N_stages
V_org_stage = V_org / N_stages
V_p_stage   = V_p   / N_stages

print("Overall reactor basis")
print("-" * 40)
print(f"V_total     = {V_total:.1f} m^3")
print(f"tau_total   = {tau_total/3600:.2f} h")
print(f"V_aq        = {V_aq:.2f} m^3")
print(f"V_org       = {V_org:.2f} m^3")
print(f"V_p         = {V_p:.2f} m^3")
print()
print("SDR in-series approximation")
print("-" * 40)
print(f"N_stages    = {N_stages}")
print(f"tau_stage   = {tau_stage:.1f} s ({tau_stage/60:.2f} min)")
print(f"V_aq_stage  = {V_aq_stage:.3f} m^3")
print(f"V_org_stage = {V_org_stage:.3f} m^3")
print(f"V_p_stage   = {V_p_stage:.3f} m^3")


### 3. SDR mass transfer basis

The slides describe the rotor–stator SDR as:
- **multistage / plug flow**,
- **high gas–liquid mass transfer**,
- **high liquid–liquid mass transfer**,
- **high liquid–solid mass transfer**.

A practical design basis is therefore taken directly from the characteristic SDR transfer ranges used in the slides:
- gas–liquid: order of **10 s⁻¹**,
- liquid–liquid: order of **300 s⁻¹**,
- liquid–solid: order of **1–300 s⁻¹**.

To keep the model transparent and stable, representative base-case values are used below.


In [ ]:
# Representative rotor–stator SDR volumetric mass-transfer coefficients
# Chosen to be clearly higher than the CSTR values, but still inside the slide-based SDR range.
kLa_GL = 10.0      # s^-1  O2 gas -> organic
kLa_LL = 300.0     # s^-1  HMF aqueous -> organic
kLa_LS_HMF = 150.0 # s^-1  HMF organic -> catalyst
kLa_LS_O2  = 300.0 # s^-1  O2 organic -> catalyst

print("SDR base-case mass transfer coefficients")
print("-" * 48)
print(f"kLa_GL     = {kLa_GL:.2f} s^-1")
print(f"kLa_LL     = {kLa_LL:.2f} s^-1")
print(f"kLa_LS_HMF = {kLa_LS_HMF:.2f} s^-1")
print(f"kLa_LS_O2  = {kLa_LS_O2:.2f} s^-1")


### 4. Reaction kinetics

The kinetic basis is copied from the CSTR notebook so the reactor concept is the only major change.


In [ ]:
# Aqueous-phase network
k1 = 0.104 / 60.0     # s^-1  Glucose -> Fructose
k2 = 0.052 / 60.0     # s^-1  Fructose -> Glucose
k3 = 0.286 / 60.0     # s^-1  Fructose -> HMF
k4 = 0.013 / 60.0     # s^-1  HMF -> Humins
k5 = 0.031 / 60.0     # s^-1  HMF -> LA + FA

# Lumped oxidation basis (same as CSTR notebook)
k_HMF_DFF   = 0.0693
k_DFF_FFCA  = 0.0273
k_HMF_HFCA  = 0.0319
k_HFCA_FFCA = 2.07e-3

k6 = 1/(1/k_HMF_DFF + 1/k_DFF_FFCA) + 1/(1/k_HMF_HFCA + 1/k_HFCA_FFCA)

# Catalyst / internal diffusion
d_p = 5e-6
phi = (d_p / 6.0) * np.sqrt(k6 / D_HMF)
eta = (1/np.tanh(3*phi) - 1/(3*phi)) / phi

print(f"k6  = {k6:.5f} m^3/mol/s")
print(f"phi = {phi:.5e}")
print(f"eta = {eta:.6f}")


### 5. O₂ saturation concentration

Henry-law basis kept the same as in the CSTR notebook.


In [ ]:
P_O2 = 10e5   # Pa oxygen partial pressure

P_aq_vap  = 10**(3.55959 - 643.748/(T - 198.043)) * 1e5
P_org_vap = 10**(3.95298 - 1254.095/(T - 71.537)) * 1e5
P_O2_min  = P_aq_vap + P_org_vap

if P_O2 < P_O2_min:
    raise ValueError("Chosen oxygen pressure is below combined vapour pressure basis.")

H_O2_298 = 101.3e3 / (8.71e-4 * (780e3 / 58.08))
H_O2 = H_O2_298 * np.exp(15e3 / R * (1/298.0 - 1/T))
C_O2_sat = P_O2 / H_O2

print(f"P_vap water  = {P_aq_vap:.1f} Pa")
print(f"P_vap MIBK   = {P_org_vap:.1f} Pa")
print(f"P_O2         = {P_O2/1e5:.1f} bar")
print(f"C*_O2        = {C_O2_sat:.4f} mol/m^3")


### 6. Feed conditions and partitioning

In [ ]:
C_Glu_feed = 1500.0   # mol/m^3 aqueous
m_AO = 0.77            # HMF partition coefficient basis

print(f"C_Glu_feed = {C_Glu_feed:.1f} mol/m^3")
print(f"m_AO       = {m_AO:.2f}")
print(f"F_aq       = {F_aq:.5f} m^3/s")
print(f"F_org      = {F_org:.5f} m^3/s")


### 7. Stage model

Each stage is treated as a small CSTR with:
- the **same kinetics** as the CSTR notebook,
- **SDR mass transfer coefficients**,
- inlet concentrations taken from the previous stage.

The full SDR is then obtained by marching stage-by-stage from inlet to outlet.


In [ ]:
def sdr_stage_odes(t, y, aq_in, org_in, pars):
    (
        k1, k2, k3, k4, k5, k6, eta,
        kLa_GL, kLa_LL, kLa_LS_HMF, kLa_LS_O2,
        m_AO, C_O2_sat,
        F_aq, F_org,
        V_aq_stage, V_org_stage, V_p_stage
    ) = pars

    Glu, Fru, HMF_aq, HMF_org, HMF_p, O2_org, O2_p, Hum, LA = [max(v, 0.0) for v in y]

    # Reaction rates
    r1 = k1 * Glu
    r2 = k2 * Fru
    r3 = k3 * Fru
    r4 = k4 * HMF_aq
    r5 = k5 * HMF_aq
    r6 = k6 * eta * HMF_p * O2_p

    # Mass transfer fluxes
    J_GL  = kLa_GL * (C_O2_sat - O2_org)
    J_LL  = kLa_LL * (HMF_aq - m_AO * HMF_org)
    J_HMF = kLa_LS_HMF * (HMF_org - HMF_p)
    J_O2  = kLa_LS_O2  * (O2_org - O2_p)

    # Mole balances
    dGlu    = (F_aq / V_aq_stage)  * (aq_in[0] - Glu)    + (-r1 + r2)
    dFru    = (F_aq / V_aq_stage)  * (aq_in[1] - Fru)    + (r1 - r2 - r3)
    dHMF_aq = (F_aq / V_aq_stage)  * (aq_in[2] - HMF_aq) + (r3 - r4 - r5) - J_LL

    dHMF_org = (F_org / V_org_stage) * (org_in[0] - HMF_org) + J_LL * (V_aq_stage / V_org_stage) - J_HMF
    dO2_org  = (F_org / V_org_stage) * (org_in[1] - O2_org)  + J_GL - J_O2

    dHMF_p = J_HMF * (V_org_stage / V_p_stage) - r6
    dO2_p  = J_O2  * (V_org_stage / V_p_stage) - r6

    dHum = (F_aq / V_aq_stage) * (aq_in[3] - Hum) + r4
    dLA  = (F_aq / V_aq_stage) * (aq_in[4] - LA)  + r5

    return [dGlu, dFru, dHMF_aq, dHMF_org, dHMF_p, dO2_org, dO2_p, dHum, dLA]


def run_sdr_model(
    N_stages=20,
    tau_total=3600.0,
    kLa_GL=10.0,
    kLa_LL=300.0,
    kLa_LS_HMF=150.0,
    kLa_LS_O2=300.0,
    eps_g=0.20,
    eps_s=0.10,
    eps_aq=1/3,
):
    eps_l = 1.0 - eps_g - eps_s
    eps_org = 1.0 - eps_aq

    V_g   = eps_g * V_total
    V_p   = eps_s * V_total
    V_liq = eps_l * V_total
    V_aq  = eps_aq * V_liq
    V_org = eps_org * V_liq

    F_aq  = V_aq / tau_total
    F_org = V_org / tau_total

    V_aq_stage  = V_aq  / N_stages
    V_org_stage = V_org / N_stages
    V_p_stage   = V_p   / N_stages
    tau_stage   = tau_total / N_stages

    pars = (
        k1, k2, k3, k4, k5, k6, eta,
        kLa_GL, kLa_LL, kLa_LS_HMF, kLa_LS_O2,
        m_AO, C_O2_sat,
        F_aq, F_org,
        V_aq_stage, V_org_stage, V_p_stage
    )

    aq_in = np.array([C_Glu_feed, 0.0, 0.0, 0.0, 0.0], dtype=float)
    org_in = np.array([0.0, C_O2_sat], dtype=float)

    stage_ss = []
    stage_dyn = []
    r6_stage = []

    for i in range(N_stages):
        y0 = np.array([
            aq_in[0], aq_in[1], aq_in[2],
            org_in[0], 0.0,
            org_in[1], org_in[1],
            aq_in[3], aq_in[4]
        ], dtype=float)

        sol = solve_ivp(
            lambda t, y: sdr_stage_odes(t, y, aq_in, org_in, pars),
            [0.0, 10.0 * tau_stage],
            y0,
            method="BDF",
            rtol=1e-8,
            atol=1e-10,
            dense_output=True
        )

        ss = sol.y[:, -1]

        stage_ss.append(ss)
        stage_dyn.append(sol)

        rate_r6 = k6 * eta * max(ss[4], 0.0) * max(ss[6], 0.0)
        r6_stage.append(rate_r6)

        aq_in = np.array([ss[0], ss[1], ss[2], ss[7], ss[8]])
        org_in = np.array([ss[3], ss[5]])

    stage_ss = np.array(stage_ss)
    r6_stage = np.array(r6_stage)

    ss_out = stage_ss[-1, :]

    F_FDCA_total = np.sum(r6_stage * V_p_stage)
    F_Hum_out = F_aq * ss_out[7]
    F_LA_out  = F_aq * ss_out[8]
    F_FA_out  = F_LA_out
    F_Glu_rxd = F_aq * (C_Glu_feed - ss_out[0])

    X_Glu = (C_Glu_feed - ss_out[0]) / C_Glu_feed

    if F_Glu_rxd > 1e-15:
        S_FDCA = F_FDCA_total / F_Glu_rxd
        S_Hum  = F_Hum_out / F_Glu_rxd
        S_LA   = F_LA_out  / F_Glu_rxd
    else:
        S_FDCA = 0.0
        S_Hum  = 0.0
        S_LA   = 0.0

    results = {
        "N_stages": N_stages,
        "tau_total": tau_total,
        "tau_stage": tau_stage,
        "F_aq": F_aq,
        "F_org": F_org,
        "V_aq": V_aq,
        "V_org": V_org,
        "V_p": V_p,
        "V_aq_stage": V_aq_stage,
        "V_org_stage": V_org_stage,
        "V_p_stage": V_p_stage,
        "stage_ss": stage_ss,
        "stage_dyn": stage_dyn,
        "r6_stage": r6_stage,
        "outlet": ss_out,
        "F_FDCA_total": F_FDCA_total,
        "F_Hum_out": F_Hum_out,
        "F_LA_out": F_LA_out,
        "F_FA_out": F_FA_out,
        "F_Glu_rxd": F_Glu_rxd,
        "X_Glu": X_Glu,
        "S_FDCA": S_FDCA,
        "S_Hum": S_Hum,
        "S_LA": S_LA,
        "kLa_GL": kLa_GL,
        "kLa_LL": kLa_LL,
        "kLa_LS_HMF": kLa_LS_HMF,
        "kLa_LS_O2": kLa_LS_O2,
        "eps_g": eps_g,
        "eps_s": eps_s,
        "eps_aq": eps_aq,
    }
    return results


### 8. Base-case SDR run

In [ ]:
res = run_sdr_model(
    N_stages=N_stages,
    tau_total=tau_total,
    kLa_GL=kLa_GL,
    kLa_LL=kLa_LL,
    kLa_LS_HMF=kLa_LS_HMF,
    kLa_LS_O2=kLa_LS_O2,
    eps_g=eps_g,
    eps_s=eps_s,
    eps_aq=eps_aq,
)

stage_ss = res["stage_ss"]
ss = res["outlet"]

Glu_ss, Fru_ss, HMFaq_ss, HMForg_ss, HMFp_ss, O2org_ss, O2p_ss, Hum_ss, LA_ss = ss

print("Base-case SDR outlet concentrations")
print("-" * 48)
print(f"Glu     = {Glu_ss:.3f} mol/m^3")
print(f"Fru     = {Fru_ss:.3f} mol/m^3")
print(f"HMF_aq  = {HMFaq_ss:.5f} mol/m^3")
print(f"HMF_org = {HMForg_ss:.5f} mol/m^3")
print(f"HMF_p   = {HMFp_ss:.5f} mol/m^3")
print(f"O2_org  = {O2org_ss:.5f} mol/m^3")
print(f"O2_p    = {O2p_ss:.5f} mol/m^3")
print(f"Humins  = {Hum_ss:.5f} mol/m^3")
print(f"LA = FA = {LA_ss:.5f} mol/m^3")


### 9. Performance metrics

In [ ]:
MW_FDCA = 168.11
MW_Hum = 126.11
MW_LA = 116.12
MW_FA = 46.03

FDCA_kgh = res["F_FDCA_total"] * MW_FDCA / 1000.0 * 3600.0
Hum_kgh  = res["F_Hum_out"]    * MW_Hum  / 1000.0 * 3600.0
LA_kgh   = res["F_LA_out"]     * MW_LA   / 1000.0 * 3600.0
FA_kgh   = res["F_FA_out"]     * MW_FA   / 1000.0 * 3600.0

print("=" * 56)
print("SDR BASE-CASE RESULTS")
print("=" * 56)
print(f"Number of stages            = {res['N_stages']}")
print(f"Total residence time        = {res['tau_total']/3600:.2f} h")
print(f"Residence time per stage    = {res['tau_stage']:.1f} s")
print()
print(f"Glucose conversion          = {res['X_Glu']*100:.2f} %")
print(f"FDCA selectivity            = {res['S_FDCA']*100:.2f} %")
print(f"Humins selectivity          = {res['S_Hum']*100:.3f} %")
print(f"LA+FA selectivity           = {res['S_LA']*100:.3f} %")
print()
print(f"FDCA production             = {FDCA_kgh:.1f} kg/h")
print(f"Humins production           = {Hum_kgh:.3f} kg/h")
print(f"LA production               = {LA_kgh:.3f} kg/h")
print(f"FA production               = {FA_kgh:.3f} kg/h")


### 10. Stagewise profiles

The plots below show how the concentrations evolve from stage 1 to stage N.  
This is the SDR analogue of moving along the flow path in a plug-flow reactor.


In [ ]:
stage_index = np.arange(1, res["N_stages"] + 1)
fdca_stage_kgh = res["r6_stage"] * res["V_p_stage"] * MW_FDCA / 1000.0 * 3600.0
fdca_cum_kgh = np.cumsum(fdca_stage_kgh)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    "SDR (N CSTRs in series): stagewise concentration and performance profiles",
    fontsize=12, fontweight="bold"
)
plt.subplots_adjust(hspace=0.38, wspace=0.30, top=0.90)

# Sugars
ax = axes[0, 0]
ax.plot(stage_index, stage_ss[:, 0], label="Glucose")
ax.plot(stage_index, stage_ss[:, 1], label="Fructose")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("Sugars (aqueous)")
ax.grid(alpha=0.3)
ax.legend()

# HMF
ax = axes[0, 1]
ax.plot(stage_index, stage_ss[:, 2], label="HMF (aq)")
ax.plot(stage_index, stage_ss[:, 3], label="HMF (org)")
ax.plot(stage_index, stage_ss[:, 4], "--", label="HMF (particle)")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("HMF across phases")
ax.grid(alpha=0.3)
ax.legend()

# O2
ax = axes[0, 2]
ax.plot(stage_index, stage_ss[:, 5], label="O₂ (org)")
ax.plot(stage_index, stage_ss[:, 6], "--", label="O₂ (particle)")
ax.axhline(C_O2_sat, color="steelblue", ls=":", lw=1, label=f"C* = {C_O2_sat:.2f}")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("O₂ profiles")
ax.grid(alpha=0.3)
ax.legend()

# Byproducts
ax = axes[1, 0]
ax.plot(stage_index, stage_ss[:, 7], label="Humins")
ax.plot(stage_index, stage_ss[:, 8], "--", label="LA = FA")
ax.set_xlabel("Stage number")
ax.set_ylabel("Concentration (mol/m³)")
ax.set_title("Byproducts (aqueous)")
ax.grid(alpha=0.3)
ax.legend()

# Conversion / selectivity
ax = axes[1, 1]
X_stage = (C_Glu_feed - stage_ss[:, 0]) / C_Glu_feed * 100.0
F_Glu_rxd_stage = res["F_aq"] * (C_Glu_feed - stage_ss[:, 0])
S_stage = np.where(
    F_Glu_rxd_stage > 1e-15,
    np.cumsum(res["r6_stage"] * res["V_p_stage"]) / F_Glu_rxd_stage * 100.0,
    0.0,
)
ax2 = ax.twinx()
l1, = ax.plot(stage_index, X_stage, label="Glucose conversion (%)")
l2, = ax2.plot(stage_index, S_stage, "C3--", label="FDCA selectivity (%)")
ax.set_xlabel("Stage number")
ax.set_ylabel("Conversion (%)")
ax2.set_ylabel("Selectivity (%)")
ax.set_ylim(0, 105)
ax2.set_ylim(0, 105)
ax.set_title("Performance")
ax.grid(alpha=0.3)
ax.legend([l1, l2], [l1.get_label(), l2.get_label()], loc="center right")

# FDCA generation
ax = axes[1, 2]
ax.plot(stage_index, fdca_stage_kgh, label="FDCA generated in stage")
ax.plot(stage_index, fdca_cum_kgh, "--", label="Cumulative FDCA")
ax.set_xlabel("Stage number")
ax.set_ylabel("kg/h")
ax.set_title("FDCA production")
ax.grid(alpha=0.3)
ax.legend()

plt.show()


### 11. Rate-determining-step (RDS) analysis

As in the CSTR notebook, the maximum possible rate of each step is estimated using the maximum driving force.
The smallest maximum rate is the bottleneck.


In [ ]:
K_eq = k1 / k2
C_Fru_max = K_eq / (1.0 + K_eq) * C_Glu_feed
C_HMF_aq_max = C_Glu_feed
C_HMF_org_max = m_AO * C_HMF_aq_max

rds = {
    "k1  Glu->Fru  (aq)": k1 * C_Glu_feed * res["V_aq"],
    "k3  Fru->HMF  (aq)": k3 * C_Fru_max * res["V_aq"],
    "LL  HMF aq->org":    res["kLa_LL"] * C_HMF_aq_max * res["V_aq"],
    "LS  HMF org->cat":   res["kLa_LS_HMF"] * C_HMF_org_max * res["V_org"],
    "GL  O2  gas->org":   res["kLa_GL"] * C_O2_sat * res["V_org"],
    "LS  O2  org->cat":   res["kLa_LS_O2"] * C_O2_sat * res["V_org"],
    "k6*eta reaction":    k6 * eta * C_HMF_org_max * C_O2_sat * res["V_p"],
}

min_rate = min(rds.values())

print("=" * 58)
print("SDR RDS -- MAXIMUM DRIVING FORCE [mol/s]")
print("=" * 58)
for name, rate in rds.items():
    tag = "  <-- BOTTLENECK" if rate == min_rate else ""
    print(f"{name:24s} {rate:12.3e}{tag}")


### 12. Optional sensitivity: number of stages

This quick check shows how the SDR approaches plug-flow behaviour as the number of stages increases.


In [ ]:
rows = []
for N in [1, 5, 10, 20, 40]:
    rr = run_sdr_model(
        N_stages=N,
        tau_total=tau_total,
        kLa_GL=kLa_GL,
        kLa_LL=kLa_LL,
        kLa_LS_HMF=kLa_LS_HMF,
        kLa_LS_O2=kLa_LS_O2,
        eps_g=eps_g,
        eps_s=eps_s,
        eps_aq=eps_aq,
    )
    rows.append({
        "N_stages": N,
        "tau_stage_s": rr["tau_stage"],
        "X_Glu_%": rr["X_Glu"] * 100.0,
        "S_FDCA_%": rr["S_FDCA"] * 100.0,
        "FDCA_kg_h": rr["F_FDCA_total"] * MW_FDCA / 1000.0 * 3600.0,
        "Humins_kg_h": rr["F_Hum_out"] * MW_Hum / 1000.0 * 3600.0,
    })

sens_df = pd.DataFrame(rows)
sens_df


### 13. Short interpretation

For this base case:
- the **SDR model gives much higher mass transfer** than the CSTR basis,
- the **stagewise / near-plug-flow behaviour** reduces the broad RTD penalty,
- the **overall bottleneck remains the first aqueous reaction step** (glucose → fructose),
- so further improvement would likely come more from **kinetics / catalyst / temperature optimisation** than from even more mass transfer.


In [ ]:
summary = {
    "X_Glu_percent": round(res["X_Glu"] * 100, 2),
    "S_FDCA_percent": round(res["S_FDCA"] * 100, 2),
    "FDCA_kg_per_h": round(FDCA_kgh, 2),
    "Humins_kg_per_h": round(Hum_kgh, 4),
    "LA_kg_per_h": round(LA_kgh, 4),
    "FA_kg_per_h": round(FA_kgh, 4),
    "RDS": min(rds, key=rds.get),
}
summary
